# 15 — Image Generation Model Prompting

In [1]:
import os, warnings
warnings.filterwarnings('ignore')
from dotenv import load_dotenv
load_dotenv(os.path.join(os.getcwd(), '..', '.env'))
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
assert GROQ_API_KEY, "GROQ_API_KEY not found — check ../.env"
print("API key loaded")


API key loaded


In [2]:
from groq import Groq
client = Groq()
MODEL = "llama-3.1-8b-instant"
MODEL_LARGE = "llama-3.3-70b-versatile"

def chat(messages, model=MODEL, temperature=0.7, max_tokens=512):
    resp = client.chat.completions.create(
        model=model, messages=messages,
        temperature=temperature, max_tokens=max_tokens
    )
    return resp.choices[0].message.content

def system_chat(system, user, **kw):
    return chat([{"role": "system", "content": system},
                 {"role": "user", "content": user}], **kw)

print(f"Groq client ready | default model: {MODEL}")


Groq client ready | default model: llama-3.1-8b-instant


## Anatomy of an Image Generation Prompt

In [3]:
def build_image_prompt(subject, style, composition, lighting, quality):
    return f"{subject}, {style}, {composition}, {lighting}, {quality}"

example = build_image_prompt(
    subject="a lone astronaut standing on Mars surface",
    style="digital art, hyperrealistic, cinematic",
    composition="wide shot, rule of thirds",
    lighting="golden hour, dramatic shadows, backlit",
    quality="8k resolution, highly detailed, trending on artstation"
)
print("Full image prompt:")
print(example)


Full image prompt:
a lone astronaut standing on Mars surface, digital art, hyperrealistic, cinematic, wide shot, rule of thirds, golden hour, dramatic shadows, backlit, 8k resolution, highly detailed, trending on artstation


## LLM-Assisted Prompt Refinement

In [4]:
weak_image_prompt = "a cat in a forest"

refine = (
    "You are an expert at writing prompts for AI image generation (DALL-E, Stable Diffusion).\n"
    "Improve this image prompt to produce a stunning, detailed, professional image.\n"
    "Add: artistic style, lighting, composition, mood, technical quality terms.\n"
    "Keep it under 100 words.\n\n"
    f"Original: \"{weak_image_prompt}\"\n\n"
    "Improved prompt:"
)
improved = chat([{"role":"user","content":refine}], temperature=0.7, max_tokens=120)
print(f"Original: {weak_image_prompt}")
print()
print(f"Improved: {improved.strip()}")


Original: a cat in a forest

Improved: "Create a stunningly detailed, high-resolution image of a majestic, sleek black cat standing regally amidst the vibrant, emerald green foliage of a misty forest, with warm golden sunlight filtering through the canopy above and casting dappled shadows on the forest floor, in the style of a classical Renaissance painting, with crisp, realistic textures and a sense of depth and atmosphere, using advanced photorealistic techniques, with a shallow depth of field to blur the background and emphasize the cat's sharp features, capturing a sense of serene calm and mystery."


## Style Transfer Prompting

In [5]:
subject = "a bustling city market"
styles = [
    "impressionist oil painting, visible brushstrokes, Monet style",
    "cyberpunk neon aesthetic, rain-soaked streets, blade runner",
    "watercolor illustration, soft pastels, children book style",
    "black and white film noir photography, 1940s, high contrast",
    "Japanese anime style, Studio Ghibli inspired, warm colors",
]

print(f"Subject: {subject}\n")
for style in styles:
    prompt = f"{subject}, {style}, highly detailed"
    desc = chat([{"role":"user","content":f"In 2 sentences, describe what this image prompt would produce:\n'{prompt}'"}],
                temperature=0.3, max_tokens=80)
    print(f"Style: {style[:50]}")
    print(f"  -> {desc.strip()}")
    print()


Subject: a bustling city market



Style: impressionist oil painting, visible brushstrokes, 
  -> This image prompt would produce a vibrant and detailed oil painting in the style of Claude Monet, depicting a bustling city market with visible brushstrokes and a high level of realism, capturing the energy and movement of the scene. The painting would feature soft, blended colors and loose, expressive brushstrokes, characteristic of Monet's impressionist style, while still showcasing the intricate details of the market scene



Style: cyberpunk neon aesthetic, rain-soaked streets, bla
  -> This image prompt would produce a highly detailed, futuristic cityscape with a cyberpunk neon aesthetic, depicting a bustling market scene amidst rain-soaked streets, reminiscent of the iconic Blade Runner movie franchise. The image would showcase a vibrant, high-tech urban environment, teeming with life and activity, with intricate details and neon lights reflecting off the wet pavement.



Style: watercolor illustration, soft pastels, children bo
  -> The image prompt would produce a vibrant, highly detailed watercolor illustration of a bustling city market, rendered in soft pastel colors reminiscent of a children's book. The scene would likely feature a multitude of colorful stalls, lively characters, and intricate details, all crafted in a whimsical style that evokes a sense of wonder and magic.



Style: black and white film noir photography, 1940s, high
  -> This image prompt would produce a high-contrast, black and white photograph of a bustling city market, capturing the gritty and atmospheric essence of 1940s film noir, with intricate details and textures that evoke a sense of nostalgia and mystery. The image would likely feature deep shadows, strong highlights, and a moody, cinematic quality that immerses the viewer in the vibrant, yet crime-ridden



Style: Japanese anime style, Studio Ghibli inspired, warm
  -> This image prompt would produce a vibrant and detailed illustration of a bustling city market, reminiscent of Studio Ghibli's whimsical style, with intricate textures and patterns, warm colors, and a sense of energy and movement. The market scene would be filled with fantastical elements, such as lanterns, street food vendors, and exotic goods, all rendered in a highly detailed and stylized manner, characteristic



## Negative Prompting

In [6]:
positive = "portrait of a person, professional headshot, soft lighting"
negative = "blurry, distorted, extra limbs, watermark, text, low quality, cartoon"

neg_explanation = chat([{"role":"user","content":f"Explain why these negative prompt terms are useful for AI image generation:\n{negative}\nBe concise, one line per term."}],
                        temperature=0, max_tokens=200)
print("Positive prompt:", positive)
print()
print("Negative prompt:", negative)
print()
print("Why each negative term helps:")
print(neg_explanation.strip())


Positive prompt: portrait of a person, professional headshot, soft lighting

Negative prompt: blurry, distorted, extra limbs, watermark, text, low quality, cartoon

Why each negative term helps:
Here's a concise explanation of each term:

1. **Blurry**: Prevents AI from generating overly sharp or realistic images that might be mistaken for real-world objects.
2. **Distorted**: Encourages AI to avoid creating images with unnatural proportions or shapes, promoting more realistic outputs.
3. **Extra limbs**: Prevents AI from generating images with unrealistic or anatomically incorrect body parts.
4. **Watermark**: Prevents AI from generating images with unwanted or distracting watermarks, logos, or text overlays.
5. **Text**: Prevents AI from generating images with unwanted or distracting text, such as captions, labels, or other written content.
6. **Low quality**: Encourages AI to generate images with a more natural, nuanced, and detailed texture, rather than overly simplistic or pixelat

## Aspect Ratio and Composition Guide

In [7]:
compositions = {
    "16:9 landscape": "wide establishing shots, landscapes, cinematic scenes",
    "1:1 square":     "portraits, product photography, Instagram-style",
    "9:16 portrait":  "mobile content, vertical social media, character focus",
    "4:3":            "classic photography, slightly wider than square",
}
print("Aspect Ratio Guide for Image Prompting:")
for ratio, use_case in compositions.items():
    print(f"  {ratio:15s}: {use_case}")

print()
composition_terms = ["rule of thirds", "centered composition", "symmetrical",
                     "close-up", "bird's eye view", "worm's eye view", "bokeh background"]
print("Composition keywords to include in prompts:")
for term in composition_terms:
    print(f"  - {term}")


Aspect Ratio Guide for Image Prompting:
  16:9 landscape : wide establishing shots, landscapes, cinematic scenes
  1:1 square     : portraits, product photography, Instagram-style
  9:16 portrait  : mobile content, vertical social media, character focus
  4:3            : classic photography, slightly wider than square

Composition keywords to include in prompts:
  - rule of thirds
  - centered composition
  - symmetrical
  - close-up
  - bird's eye view
  - worm's eye view
  - bokeh background
